# Stage 02 — Unit Sales Forecast
**Dashboard page:** MC Sales Forecast
**Tabs:** Forecast Chart · Sales Targets · Model Breakdown · Age Distribution

**Model selection:** 2-month holdout MAPE across Linear Trend / Holt ETS / ARIMA / Prophet

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
fc   = load("unit_sales_forecast.parquet")
fc["ds"] = pd.to_datetime(fc["ds"])
hist = fc[~fc["is_forecast"]].copy()
fwd  = fc[fc["is_forecast"]].copy()

print(f"Historical months : {len(hist)}")
print(f"Forecast months   : {len(fwd)}")
print(f"Forecast horizon  : {fwd['ds'].min().strftime('%Y-%m')} -> {fwd['ds'].max().strftime('%Y-%m')}")
print()
print(fc.to_string())


## Forecast Chart — Actual vs Forecast with 80% CI

In [ ]:
fig,ax = plt.subplots(figsize=(13,5))
ax.bar(hist["ds"],hist["actual"],width=20,color=PALETTE[0],alpha=0.6,label="Actual units sold")
ax.plot(fc["ds"],fc["forecast"],color=PALETTE[1],lw=2,ls="--",label="Forecast")
ax.fill_between(fc["ds"],fc["lower_80"],fc["upper_80"],alpha=0.15,color=PALETTE[1],label="80% CI")
if "target" in fc.columns:
    target_rows = fc.dropna(subset=["target"])
    ax.step(target_rows["ds"],target_rows["target"],color=PALETTE[3],lw=1.8,where="mid",label="Sales target")
ax.axvline(hist["ds"].max(),color="gray",ls=":",lw=1.2,alpha=0.7,label="Forecast start")
ax.set_title("Yamaha Sri Lanka — Monthly Motorcycle Unit Sales Forecast
"
             "(Model selected by 2-month holdout MAPE: Linear Trend vs Holt ETS vs ARIMA vs Prophet)")
ax.set_xlabel("Month"); ax.set_ylabel("Units")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{int(x):,}"))
ax.legend(fontsize=9); plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()


## Sales Targets & Target Gap

In [ ]:
target_rows = fc.dropna(subset=["target"])
if len(target_rows)>0:
    print("Sales Targets vs Forecast:")
    print(target_rows[["period","forecast","target","target_gap"]].to_string(index=False))
    fig,axes = plt.subplots(1,2,figsize=(13,4))
    axes[0].bar(target_rows["period"],target_rows["forecast"],label="Forecast",color=PALETTE[0],alpha=0.7)
    axes[0].bar(target_rows["period"],target_rows["target"],label="Target",color=PALETTE[3],alpha=0.7,width=0.4)
    axes[0].set_title("Forecast vs Sales Target (forecast months)"); axes[0].set_ylabel("Units")
    axes[0].tick_params(axis="x",rotation=45); axes[0].legend()

    gap_color = [PALETTE[1] if g<0 else PALETTE[2] for g in target_rows["target_gap"]]
    axes[1].bar(target_rows["period"],target_rows["target_gap"],color=gap_color,edgecolor="white")
    axes[1].axhline(0,color="gray",lw=1)
    axes[1].set_title("Target Gap (Forecast - Target)
Red=below target, Green=above target")
    axes[1].set_ylabel("Units"); axes[1].tick_params(axis="x",rotation=45)
    plt.tight_layout(); plt.show()
    total_gap = target_rows["target_gap"].sum()
    print(f"
Total gap over forecast period: {total_gap:,.0f} units")
else:
    print("No sales targets set. Use the dashboard to enter targets.")


## Model-Level Forecast (Module 1)

In [ ]:
try:
    m1 = load("m1_sales_forecast.parquet")
    print(f"M1 model forecast: {len(m1):,} rows")
    print("Columns:", m1.columns.tolist())
    print(m1.head(10).to_string())
    model_col = next((c for c in ["model","Model","vehicle_model"] if c in m1.columns),None)
    fc_col    = next((c for c in ["forecast","yhat","units"] if c in m1.columns),None)
    if model_col and fc_col:
        fig,ax = plt.subplots(figsize=(13,5))
        for i,mdl in enumerate(m1[model_col].unique()):
            d = m1[m1[model_col]==mdl].sort_values(m1.columns[0])
            ax.plot(range(len(d)),d[fc_col],color=PALETTE[i%len(PALETTE)],lw=2,label=mdl)
        ax.set_title("Model-Level Unit Sales Forecast (M1 module)")
        ax.set_ylabel("Units"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
except FileNotFoundError:
    print("M1 sales forecast not yet run — run Module 1 from the Pipeline page first.")


## Age Distribution (M1 module — months on market)

In [ ]:
try:
    age = load("m1_age_distribution.parquet")
    print("Age distribution columns:", age.columns.tolist())
    print(age.to_string())
    age_col  = next((c for c in ["age_months","months_on_market","age"] if c in age.columns),None)
    cnt_col  = next((c for c in ["count","units","n"] if c in age.columns),None)
    model_col= next((c for c in ["model","Model"] if c in age.columns),None)
    if age_col:
        fig,ax = plt.subplots(figsize=(11,4))
        if model_col:
            for i,mdl in enumerate(age[model_col].unique()):
                d = age[age[model_col]==mdl]
                ax.bar(d[age_col]+(i*0.15),d[cnt_col] if cnt_col else d.iloc[:,1],
                       width=0.15,color=PALETTE[i%len(PALETTE)],label=mdl,edgecolor="white")
            ax.legend(fontsize=8)
        else:
            ax.bar(age[age_col],age[cnt_col] if cnt_col else age.iloc[:,1],color=PALETTE[0],edgecolor="white")
        ax.set_title("Motorcycle Age Distribution (Months on Market)")
        ax.set_xlabel("Age (months)"); ax.set_ylabel("Count")
        plt.tight_layout(); plt.show()
except FileNotFoundError:
    print("Age distribution data not yet available.")


**Model selection rationale:** 2-month holdout MAPE. Last 2 months withheld; all candidate models fit on the rest; the model with the lowest MAPE is refit on the full series. 2-month holdout matches the lead time horizon — near-term accuracy matters most.